# OGF Check Conformed Layer | 'RTE','RTE_OTHER'

In [0]:
%sql
WITH Jul26_AggregatedData AS (
    -- Aggregates value and volume for each Global OGF Segment within each Global Market for July 2026.
    -- Also calculates market-level totals and overall grand totals using window functions in a single pass.
    SELECT
        gm.Global_Market,
        p.Global_OGF_Segment,
        SUM(f.Value_000) * ANY_VALUE(FX.ExchangeRate) AS Value_000_CHF,
        SUM(f.Volume_000) AS Volume_000,
        -- Calculate total value/volume per market using a window function (for percentage denominator)
        SUM(SUM(f.Value_000) * ANY_VALUE(FX.ExchangeRate)) OVER (PARTITION BY gm.Global_Market) AS Total_Value_000_CHF_Market,
        SUM(SUM(f.Volume_000)) OVER (PARTITION BY gm.Global_Market) AS Total_Volume_000_Market,
        -- Calculate overall total value/volume across all markets (for the 'Overall Grand Total' row)
        SUM(SUM(f.Value_000) * ANY_VALUE(FX.ExchangeRate)) OVER () AS Overall_Total_Value_000_CHF,
        SUM(SUM(f.Volume_000)) OVER () AS Overall_Total_Volume_000
    FROM glbl_cpw_prod.conformed.factretailsales AS f
    JOIN glbl_cpw_prod.conformed.dimproduct p ON f.LocalProductKey = p.LocalProductKey
    JOIN glbl_cpw_prod.conformed.dimmarket gm ON f.LocalMarketKey = gm.LocalMarketKey
    JOIN glbl_cpw_prod.conformed.dimperiod t ON f.LocalPeriodKey = t.LocalPeriodKey
    JOIN metadata.forex FX ON f.Database = FX.Database
    JOIN glbl_cpw_prod.adhoc.period_mat_ytd as pmt ON f.LocalPeriodKey=pmt.LocalPeriodKey
    AND f.DataProvider = pmt.DataProvider
    AND f.Database = pmt.Database
    WHERE
        gm.Global_Total_Mkt_Flag = 'Y' 
        AND p.Global_Category IN ('RTE','RTE_OTHER')
        AND LEFT(t.Local_Time_Period, 4) IN ('2022', '2023', '2024', '2025', '2026')
        AND gm.local_market NOT IN ('Colombia', 'Costa Rica', 'El Salvador', 'Guatemala', 'Honduras', 'Nicaragua', 'Panamá', 'Peru','Baltics','Brazil_Scan','France_Scan','Poland_Scan','Mexico_Scan','Greece_C&C')
        AND pmt.MAT_Flag IN ('MAT TY','MAT LY','MAT 2LY') -- Filter using MAT_Flag from dimperiod_matytd_tag
    GROUP BY
        gm.Global_Market,
        p.Global_OGF_Segment
),
Jun26_AggregatedData AS (
    -- Aggregates value and volume for each Global OGF Segment within each Global Market for February 2026 (historical data).
    -- Also calculates market-level totals and overall grand totals using window functions in a single pass.
    SELECT
        gm.Global_Market,
        p.Global_OGF_Segment,
        SUM(f.Value_000) * ANY_VALUE(FX.ExchangeRate) AS Value_000_CHF,
        SUM(f.Volume_000) AS Volume_000,
        -- Calculate total value/volume per market using a window function (for percentage denominator)
        SUM(SUM(f.Value_000) * ANY_VALUE(FX.ExchangeRate)) OVER (PARTITION BY gm.Global_Market) AS Total_Value_000_CHF_Market,
        SUM(SUM(f.Volume_000)) OVER (PARTITION BY gm.Global_Market) AS Total_Volume_000_Market,
        -- Calculate overall total value/volume across all markets (for the 'Overall Grand Total' row)
        SUM(SUM(f.Value_000) * ANY_VALUE(FX.ExchangeRate)) OVER () AS Overall_Total_Value_000_CHF,
        SUM(SUM(f.Volume_000)) OVER () AS Overall_Total_Volume_000
    FROM glbl_cpw_prod.history_conformed.factretailsales_Jun26 AS f
    JOIN glbl_cpw_prod.history_conformed.dimproduct_Jun26 p ON f.LocalProductKey = p.LocalProductKey
    JOIN glbl_cpw_prod.history_conformed.dimmarket_Jun26 gm ON f.LocalMarketKey = gm.LocalMarketKey
    JOIN glbl_cpw_prod.history_conformed.dimperiod_Jun26 t ON f.LocalPeriodKey = t.LocalPeriodKey
    JOIN metadata.forex FX ON f.Database = FX.Database
    JOIN glbl_cpw_prod.history_conformed.period_mat_ytd_Jun26 pmy ON f.LocalPeriodKey = pmy.LocalPeriodKey
    WHERE
        gm.Global_Total_Mkt_Flag = 'Y' AND
        p.Global_Category IN ('RTE', 'RTE_OTHER')
        AND LEFT(t.Local_Time_Period, 4) IN ('2022', '2023', '2023', '2024', '2025', '2026')
        AND gm.local_market NOT IN ('Baltics','Brazil_Scan','France_Scan','Poland_Scan','Mexico_Scan','Greece_C&C')
        AND pmy.MAT_Flag IN ('MAT TY','MAT LY','MAT 2LY') -- Filter using MAT_Flag
    GROUP BY
        gm.Global_Market,
        p.Global_OGF_Segment
)
-- Final SELECT statement to combine the results, calculate percentages and differences, and add a grand total row.
SELECT
    COALESCE(apr.Global_Market, mar.Global_Market) AS `Global_Market`,
    COALESCE(apr.Global_OGF_Segment, mar.Global_OGF_Segment) AS `Global_OGF_Segment`,
    -- April 2026 Percentages (calculated relative to market total)
    CONCAT(ROUND((COALESCE(apr.Value_000_CHF, 0) * 100.0 / COALESCE(apr.Total_Value_000_CHF_Market, 1)), 2), '%') AS `Sum_of_Value_000_CHF_Jul26`,
    CONCAT(ROUND((COALESCE(apr.Volume_000, 0) * 100.0 / COALESCE(apr.Total_Volume_000_Market, 1)), 2), '%') AS `Sum_of_Volume_000_Jul26`,
    -- March 2026 Percentages (calculated relative to market total)
    CONCAT(ROUND((COALESCE(mar.Value_000_CHF, 0) * 100.0 / COALESCE(mar.Total_Value_000_CHF_Market, 1)), 2), '%') AS `Sum_of_Value_000_CHF_Jun26`,
    CONCAT(ROUND((COALESCE(mar.Volume_000, 0) * 100.0 / COALESCE(mar.Total_Volume_000_Market, 1)), 2), '%') AS `Sum_of_Volume_000_Jun26`,
    -- Value difference July26 vs June26 (difference in percentages)
    CONCAT(ROUND(
        (COALESCE(apr.Value_000_CHF, 0) * 100.0 / COALESCE(apr.Total_Value_000_CHF_Market, 1)) -
        (COALESCE(mar.Value_000_CHF, 0) * 100.0 / COALESCE(mar.Total_Value_000_CHF_Market, 1)), 2), '%') AS `Value difference Jul26 vs Jun26`,
    -- Volume difference July26 vs June26 (difference in percentages)
    CONCAT(ROUND(
        (COALESCE(apr.Volume_000, 0) * 100.0 / COALESCE(apr.Total_Volume_000_Market, 1)) -
        (COALESCE(mar.Volume_000, 0) * 100.0 / COALESCE(mar.Total_Volume_000_Market, 1)), 2), '%') AS `Volume difference Jul26 vs Jun26`
FROM
    Jul26_AggregatedData apr
FULL OUTER JOIN
    Jun26_AggregatedData mar ON apr.Global_OGF_Segment = mar.Global_OGF_Segment
    AND apr.Global_Market = mar.Global_Market
UNION ALL
-- Overall Grand Total row (across all markets and segments)
SELECT
    'Overall' AS `Global_Market`,
    'Grand Total' AS `Global_OGF_Segment`,
    -- July 2026 Overall Percentages (always 100% for the total)
    '100.00%' AS `Sum_of_Value_000_CHF_Jul26`,
    '100.00%' AS `Sum_of_Volume_000_Jul26`,
    -- June 2026 Overall Percentages (always 100% for the total)
    '100.00%' AS `Sum_of_Value_000_CHF_Jun26`,
    '100.00%' AS `Sum_of_Volume_000_Jun26`,
    -- Overall Value difference (calculated from overall totals)
    CONCAT(ROUND(
        (COALESCE(MAX(apr.Overall_Total_Value_000_CHF), 0) * 100.0 / COALESCE(MAX(apr.Overall_Total_Value_000_CHF), 1)) -
        (COALESCE(MAX(mar.Overall_Total_Value_000_CHF), 0) * 100.0 / COALESCE(MAX(mar.Overall_Total_Value_000_CHF), 1)), 2), '%') AS `Value difference Jul26 vs Jun26`,
    -- Overall Volume difference (calculated from overall totals)
    CONCAT(ROUND(
        (COALESCE(MAX(apr.Overall_Total_Volume_000), 0) * 100.0 / COALESCE(MAX(apr.Overall_Total_Volume_000), 1)) -
        (COALESCE(MAX(mar.Overall_Total_Volume_000), 0) * 100.0 / COALESCE(MAX(mar.Overall_Total_Volume_000), 1)), 2), '%') AS `Volume difference Jul26 vs Jun26`
FROM
    Jul26_AggregatedData apr
FULL OUTER JOIN
    Jun26_AggregatedData mar ON apr.Global_OGF_Segment = mar.Global_OGF_Segment
    AND apr.Global_Market = mar.Global_Market
ORDER BY
    CASE WHEN `Global_OGF_Segment` = 'Grand Total' THEN 2 ELSE 1 END, -- Ensures 'Grand Total' row appears last
    `Global_Market`,
    `Global_OGF_Segment`;

Global_Market,Global_OGF_Segment,Sum_of_Value_000_CHF_Jul26,Sum_of_Volume_000_Jul26,Sum_of_Value_000_CHF_Jun26,Sum_of_Volume_000_Jun26,Value difference Jul26 vs Jun26,Volume difference Jul26 vs Jun26
Australia,CHILDHOOD FUN,10.79%,7.20%,10.81%,7.22%,-0.02%,-0.01%
Australia,EVERYDAY WELLNESS,17.01%,13.97%,16.87%,13.83%,0.14%,0.14%
Australia,NATURALLY DELICIOUS,19.43%,16.60%,19.42%,16.62%,0.0%,-0.02%
Australia,SIMPLE GOODNESS,38.67%,51.53%,38.79%,51.63%,-0.12%,-0.10%
Australia,TASTY FAVOURITES,14.05%,10.68%,14.05%,10.70%,-0.01%,-0.01%
Australia,UNCLASSIFIED,0.06%,0.02%,0.05%,0.01%,0.0%,0.00%
Austria,CHILDHOOD FUN,31.77%,31.02%,31.69%,30.92%,0.08%,0.10%
Austria,EVERYDAY WELLNESS,9.15%,7.36%,9.12%,7.32%,0.03%,0.04%
Austria,NATURALLY DELICIOUS,0.69%,0.50%,0.7%,0.50%,0.0%,0.00%
Austria,SIMPLE GOODNESS,12.68%,16.66%,12.73%,16.71%,-0.05%,-0.06%


# OGF Check Derived Layer | 'RTE','RTE_OTHER'

In [0]:
%sql
WITH Jul26_SegmentAggregates AS (
    -- This CTE calculates the aggregated value and volume for each Global OGF Segment and Database for the March 2026 period.
    SELECT
        rit.Market as Market,
        rit.OGF_Segment as OGF_Segment,
        sum(rit.Value_000_CHF) as Value_000_CHF,
        sum(rit.Volume_000) as Volume_000
    FROM
        glbl_cpw_prod.derived.retailindextotal AS rit
    WHERE
        rit.Category IN ('RTE','RTE_OTHER')
        AND rit.Perd_Cal_Yr IN ('2022', '2023', '2024', '2025', '2026')
        and rit.market NOT IN('Estonia','Latvia','Lithuania','Greece_CC')
        AND rit.MAT_Flag IN ('MAT TY','MAT LY','MAT 2LY')
    GROUP BY
         rit.Market,
         rit.OGF_Segment
),
Jul26_DatabaseTotals AS ( -- Renamed from Jun26_GrandTotals to reflect per-database totals
    -- This CTE calculates the total value and volume for each Database for the March 2026 period.
    SELECT
        Market, -- Group by Market
        SUM(Value_000_CHF) AS Total_Value_000_CHF,
        SUM(Volume_000) AS Total_Volume_000
    FROM
        Jul26_SegmentAggregates
    GROUP BY
        Market -- Group by Market
),
Jul26_Percentages AS (
    -- This CTE calculates the percentage contribution of each Global OGF Segment *within its Database* for March 2026.
    SELECT
        sa.Market,
        sa.OGF_Segment,
        (sa.Value_000_CHF * 100.0 / dt.Total_Value_000_CHF) AS Raw_Value_Percentage_Jul26, -- Uses per-database total
        (sa.Volume_000 * 100.0 / dt.Total_Volume_000) AS Raw_Volume_Percentage_Jul26, -- Uses per-database total
        CONCAT(ROUND((sa.Value_000_CHF * 100.0 / dt.Total_Value_000_CHF), 2), '%') AS `Sum_of_Value_000_CHF_Jul26`,
        CONCAT(ROUND((sa.Volume_000 * 100.0 / dt.Total_Volume_000), 2), '%') AS `Sum_of_Volume_000_Jul26`
    FROM
        Jul26_SegmentAggregates sa
    INNER JOIN -- Changed to INNER JOIN to link segments to their database totals
        Jul26_DatabaseTotals dt ON sa.Market = dt.Market
),
Jun26_SegmentAggregates AS (
    -- This CTE is similar to Jun26_SegmentAggregates but for the February 2026 period, using historical data tables.
    SELECT
        rit.Market as Market,
        rit.OGF_Segment as OGF_Segment,
        sum(rit.Value_000_CHF) as Value_000_CHF,
        sum(rit.Volume_000) as Volume_000
    FROM
        glbl_cpw_prod.history_conformed.retailindextotal_Jun26 AS rit
     WHERE
        rit.Category IN ('RTE', 'RTE_OTHER')
        AND rit.Perd_Cal_Yr IN ('2022', '2023', '2024', '2025', '2026')
        and rit.market NOT IN('Estonia','Latvia','Lithuania','Greece_CC')
        AND rit.MAT_Flag IN ('MAT TY','MAT LY','MAT 2LY')
    GROUP BY
         rit.Market,
         rit.OGF_Segment
),
Jun26_DatabaseTotals AS ( -- Renamed from Jun26_GrandTotals to reflect per-database totals
    -- This CTE calculates the total value and volume for each Database for the February 2026 period.
    SELECT
        Market, -- Group by Market
        SUM(Value_000_CHF) AS Total_Value_000_CHF,
        SUM(Volume_000) AS Total_Volume_000
    FROM
        Jun26_SegmentAggregates
    GROUP BY
        Market -- Group by Database
),
Jun26_Percentages AS (
    -- This CTE calculates the percentage contribution of each Global OGF Segment *within its Database* for February 2026.
    SELECT
        sa.Market,
        sa.OGF_Segment,
        (sa.Value_000_CHF * 100.0 / dt.Total_Value_000_CHF) AS Raw_Value_Percentage_Jun26, -- Uses per-database total
        (sa.Volume_000 * 100.0 / dt.Total_Volume_000) AS Raw_Volume_Percentage_Jun26, -- Uses per-database total
        CONCAT(ROUND((sa.Value_000_CHF * 100.0 / dt.Total_Value_000_CHF), 2), '%') AS `Sum_of_Value_000_CHF_Jun26`,
        CONCAT(ROUND((sa.Volume_000 * 100.0 / dt.Total_Volume_000), 2), '%') AS `Sum_of_Volume_000_Jun26`
    FROM
        Jun26_SegmentAggregates sa
    INNER JOIN -- Changed to INNER JOIN to link segments to their database totals
        Jun26_DatabaseTotals dt ON sa.Market = dt.Market
),
Overall_GrandTotals AS ( -- New CTE for overall grand totals across all databases
    SELECT
        SUM(Value_000_CHF) AS Overall_Total_Value_000_CHF_Jul26,
        SUM(Volume_000) AS Overall_Total_Volume_000_Jul26
    FROM
        Jul26_SegmentAggregates
),
Overall_GrandTotals_Feb AS ( -- New CTE for overall grand totals across all databases for Feb
    SELECT
        SUM(Value_000_CHF) AS Overall_Total_Value_000_CHF_Jun26,
        SUM(Volume_000) AS Overall_Total_Volume_000_Jun26
    FROM
        Jun26_SegmentAggregates
)
-- Final SELECT statement to combine the results from March and February 2026 percentages,
-- add a grand total row, and calculate the differences.
SELECT
    COALESCE(mp.Market, fp.Market) AS `Market`,
    COALESCE(mp.OGF_Segment, fp.OGF_Segment) AS `Global_OGF_Segment`,
    mp.`Sum_of_Value_000_CHF_Jul26`,
    mp.`Sum_of_Volume_000_Jul26`,
    fp.`Sum_of_Value_000_CHF_Jun26`,
    fp.`Sum_of_Volume_000_Jun26`,
    CONCAT(ROUND(COALESCE(mp.Raw_Value_Percentage_Jul26, 0) - COALESCE(fp.Raw_Value_Percentage_Jun26, 0), 2), '%') AS `Value difference Jul26 vs Jun26`,
    CONCAT(ROUND(COALESCE(mp.Raw_Volume_Percentage_Jul26, 0) - COALESCE(fp.Raw_Volume_Percentage_Jun26, 0), 2), '%') AS `Volume difference Jul26 vs Jun26`
FROM
    Jul26_Percentages mp
FULL OUTER JOIN
    Jun26_Percentages fp ON mp.OGF_Segment = fp.OGF_Segment
    AND mp.Market = fp.Market
UNION ALL
-- Grand Total row for overall percentages (across all databases)
SELECT
    'Overall' AS `Market`, -- Label for the overall total
    'Grand Total' AS `Global_OGF_Segment`,
    CONCAT(ROUND((SUM(sa.Value_000_CHF) * 100.0 / ogt.Overall_Total_Value_000_CHF_Jul26), 2), '%') AS `Sum_of_Value_000_CHF_Jul26`,
    CONCAT(ROUND((SUM(sa.Volume_000) * 100.0 / ogt.Overall_Total_Volume_000_Jul26), 2), '%') AS `Sum_of_Volume_000_Jun26`,
    CONCAT(ROUND((SUM(sa_mar.Value_000_CHF) * 100.0 / ogt_mar.Overall_Total_Value_000_CHF_Jun26), 2), '%') AS `Sum_of_Value_000_CHF_Jun26`,
    CONCAT(ROUND((SUM(sa_mar.Volume_000) * 100.0 / ogt_mar.Overall_Total_Volume_000_Jun26), 2), '%') AS `Sum_of_Volume_000_Jun26`,
    CONCAT(ROUND(
        (SUM(sa.Value_000_CHF) * 100.0 / ogt.Overall_Total_Value_000_CHF_Jul26) -
        (SUM(sa_mar.Value_000_CHF) * 100.0 / ogt_mar.Overall_Total_Value_000_CHF_Jun26), 2), '%') AS `Value difference Jul26 vs Jun26`,
    CONCAT(ROUND(
        (SUM(sa.Volume_000) * 100.0 / ogt.Overall_Total_Volume_000_Jul26) -
        (SUM(sa_mar.Volume_000) * 100.0 / ogt_mar.Overall_Total_Volume_000_Jun26), 2), '%') AS `Volume difference Jul26 vs Jun26`
FROM
    Jun26_SegmentAggregates sa
CROSS JOIN Overall_GrandTotals ogt
CROSS JOIN Jun26_SegmentAggregates sa_mar
CROSS JOIN Overall_GrandTotals_Feb ogt_mar
GROUP BY
    ogt.Overall_Total_Value_000_CHF_Jul26, ogt.Overall_Total_Volume_000_Jul26,
    ogt_mar.Overall_Total_Value_000_CHF_Jun26, ogt_mar.Overall_Total_Volume_000_Jun26
ORDER BY
    CASE WHEN `Global_OGF_Segment` = 'Grand Total' THEN 2 ELSE 1 END,
    `Global_OGF_Segment`,
    `Market`;

Market,Global_OGF_Segment,Sum_of_Value_000_CHF_Jul26,Sum_of_Volume_000_Jul26,Sum_of_Value_000_CHF_Jun26,Sum_of_Volume_000_Jun26,Value difference Jul26 vs Jun26,Volume difference Jul26 vs Jun26
Australia,CHILDHOOD FUN,10.81%,7.21%,10.83%,7.22%,-0.02%,-0.01%
Austria,CHILDHOOD FUN,31.79%,31.03%,31.71%,30.93%,0.08%,0.10%
Austria_Muesli,CHILDHOOD FUN,0.60%,0.43%,0.59%,0.43%,0.01%,0.00%
Brazil,CHILDHOOD FUN,4.48%,3.96%,4.51%,3.98%,-0.03%,-0.02%
Bulgaria,CHILDHOOD FUN,31.12%,28.58%,31.09%,28.56%,0.03%,0.03%
Chile,CHILDHOOD FUN,20.72%,17.80%,20.74%,17.85%,-0.03%,-0.06%
Czech,CHILDHOOD FUN,20.86%,20.20%,20.17%,19.16%,0.68%,1.04%
France,CHILDHOOD FUN,20.12%,19.01%,20.04%,18.92%,0.09%,0.09%
Germany,CHILDHOOD FUN,14.93%,14.22%,14.93%,14.22%,0.00%,-0.01%
Greece,CHILDHOOD FUN,23.99%,25.70%,24.03%,25.77%,-0.04%,-0.07%


# Manufacturer Check Conformed Layer | 'RTE','RTE_OTHER'

In [0]:
%sql
WITH Jul26_AggregatedData AS (
    -- Aggregates value and volume for each Global OGF Segment within each Global Market for March 2026.
    -- Also calculates market-level totals and overall grand totals using window functions in a single pass.
    SELECT
        gm.Global_Market,
        p.Global_Manufacturer,
        SUM(f.Value_000) * ANY_VALUE(FX.ExchangeRate) AS Value_000_CHF,
        SUM(f.Volume_000) AS Volume_000,
        -- Calculate total value/volume per market using a window function (for percentage denominator)
        SUM(SUM(f.Value_000) * ANY_VALUE(FX.ExchangeRate)) OVER (PARTITION BY gm.Global_Market) AS Total_Value_000_CHF_Market,
        SUM(SUM(f.Volume_000)) OVER (PARTITION BY gm.Global_Market) AS Total_Volume_000_Market,
        -- Calculate overall total value/volume across all markets (for the 'Overall Grand Total' row)
        SUM(SUM(f.Value_000) * ANY_VALUE(FX.ExchangeRate)) OVER () AS Overall_Total_Value_000_CHF,
        SUM(SUM(f.Volume_000)) OVER () AS Overall_Total_Volume_000
    FROM glbl_cpw_prod.conformed.factretailsales AS f
    JOIN glbl_cpw_prod.conformed.dimproduct p ON f.LocalProductKey = p.LocalProductKey
    JOIN glbl_cpw_prod.conformed.dimmarket gm ON f.LocalMarketKey = gm.LocalMarketKey
    JOIN glbl_cpw_prod.conformed.dimperiod t ON f.LocalPeriodKey = t.LocalPeriodKey
    JOIN metadata.forex FX ON f.Database = FX.Database
    JOIN glbl_cpw_prod.adhoc.period_mat_ytd as pmt ON f.LocalPeriodKey=pmt.LocalPeriodKey
    AND f.DataProvider = pmt.DataProvider
    AND f.Database = pmt.Database
    WHERE
        gm.Global_Total_Mkt_Flag = 'Y' 
        AND p.Global_Category IN ('RTE','RTE_OTHER')
        AND LEFT(t.Local_Time_Period, 4) IN ('2022', '2023', '2024', '2025', '2026')
        AND gm.local_market NOT IN ('Colombia', 'Costa Rica', 'El Salvador', 'Guatemala', 'Honduras', 'Nicaragua', 'Panamá', 'Peru','Baltics','Brazil_Scan','France_Scan','Poland_Scan','Mexico_Scan','Greece_C&C')
        AND pmt.MAT_Flag IN ('MAT TY','MAT LY','MAT 2LY') -- Filter using MAT_Flag from dimperiod_matytd_tag
    GROUP BY
        gm.Global_Market,
        p.Global_Manufacturer
),
Jun26_AggregatedData AS (
    -- Aggregates value and volume for each Global OGF Segment within each Global Market for February 2026 (historical data).
    -- Also calculates market-level totals and overall grand totals using window functions in a single pass.
    SELECT
        gm.Global_Market,
        p.Global_Manufacturer,
        SUM(f.Value_000) * ANY_VALUE(FX.ExchangeRate) AS Value_000_CHF,
        SUM(f.Volume_000) AS Volume_000,
        -- Calculate total value/volume per market using a window function (for percentage denominator)
        SUM(SUM(f.Value_000) * ANY_VALUE(FX.ExchangeRate)) OVER (PARTITION BY gm.Global_Market) AS Total_Value_000_CHF_Market,
        SUM(SUM(f.Volume_000)) OVER (PARTITION BY gm.Global_Market) AS Total_Volume_000_Market,
        -- Calculate overall total value/volume across all markets (for the 'Overall Grand Total' row)
        SUM(SUM(f.Value_000) * ANY_VALUE(FX.ExchangeRate)) OVER () AS Overall_Total_Value_000_CHF,
        SUM(SUM(f.Volume_000)) OVER () AS Overall_Total_Volume_000
    FROM glbl_cpw_prod.history_conformed.factretailsales_Jun26 AS f
    JOIN glbl_cpw_prod.history_conformed.dimproduct_Jun26 p ON f.LocalProductKey = p.LocalProductKey
    JOIN glbl_cpw_prod.history_conformed.dimmarket_Jun26 gm ON f.LocalMarketKey = gm.LocalMarketKey
    JOIN glbl_cpw_prod.history_conformed.dimperiod_Jun26 t ON f.LocalPeriodKey = t.LocalPeriodKey
    JOIN metadata.forex FX ON f.Database = FX.Database
    JOIN glbl_cpw_prod.history_conformed.period_mat_ytd_Jun26 pmy ON f.LocalPeriodKey = pmy.LocalPeriodKey
    WHERE
        gm.Global_Total_Mkt_Flag = 'Y' AND
        p.Global_Category IN ('RTE', 'RTE_OTHER')
        AND LEFT(t.Local_Time_Period, 4) IN ('2022', '2023', '2023', '2024', '2025', '2026')
        AND gm.local_market NOT IN ('Baltics','Brazil_Scan','France_Scan','Poland_Scan','Mexico_Scan','Greece_C&C')
        AND pmy.MAT_Flag IN ('MAT TY','MAT LY','MAT 2LY') -- Filter using MAT_Flag
    GROUP BY
        gm.Global_Market,
        p.Global_Manufacturer
)
-- Final SELECT statement to combine the results, calculate percentages and differences, and add a grand total row.
SELECT
    COALESCE(apr.Global_Market, mar.Global_Market) AS `Global_Market`,
    COALESCE(apr.Global_Manufacturer, mar.Global_Manufacturer) AS `Global_Manufacturer`,
    -- July 2026 Percentages (calculated relative to market total)
    CONCAT(ROUND((COALESCE(apr.Value_000_CHF, 0) * 100.0 / COALESCE(apr.Total_Value_000_CHF_Market, 1)), 2), '%') AS `Sum_of_Value_000_CHF_Jul26`,
    CONCAT(ROUND((COALESCE(apr.Volume_000, 0) * 100.0 / COALESCE(apr.Total_Volume_000_Market, 1)), 2), '%') AS `Sum_of_Volume_000_Jul26`,
    -- June 2026 Percentages (calculated relative to market total)
    CONCAT(ROUND((COALESCE(mar.Value_000_CHF, 0) * 100.0 / COALESCE(mar.Total_Value_000_CHF_Market, 1)), 2), '%') AS `Sum_of_Value_000_CHF_Jun26`,
    CONCAT(ROUND((COALESCE(mar.Volume_000, 0) * 100.0 / COALESCE(mar.Total_Volume_000_Market, 1)), 2), '%') AS `Sum_of_Volume_000_Jun26`,
    -- Value difference July26 vs June26 (difference in percentages)
    CONCAT(ROUND(
        (COALESCE(apr.Value_000_CHF, 0) * 100.0 / COALESCE(apr.Total_Value_000_CHF_Market, 1)) -
        (COALESCE(mar.Value_000_CHF, 0) * 100.0 / COALESCE(mar.Total_Value_000_CHF_Market, 1)), 2), '%') AS `Value difference Jul26 vs Jun26`,
    -- Volume difference July26 vs June26 (difference in percentages)
    CONCAT(ROUND(
        (COALESCE(apr.Volume_000, 0) * 100.0 / COALESCE(apr.Total_Volume_000_Market, 1)) -
        (COALESCE(mar.Volume_000, 0) * 100.0 / COALESCE(mar.Total_Volume_000_Market, 1)), 2), '%') AS `Volume difference Jul26 vs Jun26`
FROM
    Jul26_AggregatedData apr
FULL OUTER JOIN
    Jun26_AggregatedData mar ON apr.Global_Manufacturer = mar.Global_Manufacturer
    AND apr.Global_Market = mar.Global_Market
UNION ALL
-- Overall Grand Total row (across all markets and segments)
SELECT
    'Overall' AS `Global_Market`,
    'Grand Total' AS `Global_Manufacturer`,
    -- April 2026 Overall Percentages (always 100% for the total)
    '100.00%' AS `Sum_of_Value_000_CHF_Jul26`,
    '100.00%' AS `Sum_of_Volume_000_Jul26`,
    -- February 2026 Overall Percentages (always 100% for the total)
    '100.00%' AS `Sum_of_Value_000_CHF_Jun26`,
    '100.00%' AS `Sum_of_Volume_000_Jun26`,
    -- Overall Value difference (calculated from overall totals)
    CONCAT(ROUND(
        (COALESCE(MAX(apr.Overall_Total_Value_000_CHF), 0) * 100.0 / COALESCE(MAX(apr.Overall_Total_Value_000_CHF), 1)) -
        (COALESCE(MAX(mar.Overall_Total_Value_000_CHF), 0) * 100.0 / COALESCE(MAX(mar.Overall_Total_Value_000_CHF), 1)), 2), '%') AS `Value difference Jul26 vs Jun26`,
    -- Overall Volume difference (calculated from overall totals)
    CONCAT(ROUND(
        (COALESCE(MAX(apr.Overall_Total_Volume_000), 0) * 100.0 / COALESCE(MAX(apr.Overall_Total_Volume_000), 1)) -
        (COALESCE(MAX(mar.Overall_Total_Volume_000), 0) * 100.0 / COALESCE(MAX(mar.Overall_Total_Volume_000), 1)), 2), '%') AS `Volume difference Jul26 vs Jun26`
FROM
    Jul26_AggregatedData apr
FULL OUTER JOIN
    Jun26_AggregatedData mar ON apr.Global_Manufacturer = mar.Global_Manufacturer
    AND apr.Global_Market = mar.Global_Market
ORDER BY
    CASE WHEN `Global_Manufacturer` = 'Grand Total' THEN 2 ELSE 1 END, -- Ensures 'Grand Total' row appears last
    `Global_Market`,
    `Global_Manufacturer`;

Global_Market,Global_Manufacturer,Sum_of_Value_000_CHF_Jul26,Sum_of_Volume_000_Jul26,Sum_of_Value_000_CHF_Jun26,Sum_of_Volume_000_Jun26,Value difference Jul26 vs Jun26,Volume difference Jul26 vs Jun26
Australia,ALFONZO RIVAS,0.0%,0.00%,0.0%,0.00%,0.0%,0.00%
Australia,ALL OTHER MANUFACTURERS,0.73%,0.28%,0.73%,0.29%,0.0%,0.00%
Australia,BOBS RED MILL,0.0%,0.00%,0.0%,0.00%,0.0%,0.00%
Australia,CARMANS FINE FOODS,9.15%,5.97%,9.15%,5.98%,0.0%,-0.01%
Australia,CHANTAL ORGANIC WHOLESALERS,0.0%,0.00%,0.0%,0.00%,0.0%,0.00%
Australia,DORSET CEREALS,0.34%,0.23%,0.35%,0.24%,-0.01%,-0.01%
Australia,DR SCHAR,0.0%,0.00%,0.0%,0.00%,0.0%,0.00%
Australia,EAST BALI CASHEWS,0.0%,0.00%,0.0%,0.00%,0.0%,0.00%
Australia,ESGIR,0.0%,0.00%,0.0%,0.00%,0.0%,0.00%
Australia,FARMER JO,0.3%,0.08%,0.3%,0.09%,0.0%,0.00%


# Manufacturer Check Derived Layer | 'RTE','RTE_OTHER'

In [0]:
%sql
WITH Jul26_SegmentAggregates AS (
    -- This CTE calculates the aggregated value and volume for each Global OGF Segment and Database for the March 2026 period.
    SELECT
        rit.Market as Market,
        rit.Manufacturer as Manufacturer,
        sum(rit.Value_000_CHF) as Value_000_CHF,
        sum(rit.Volume_000) as Volume_000
    FROM
        glbl_cpw_prod.derived.retailindextotal AS rit
    WHERE
        rit.Category IN ('RTE','RTE_OTHER')
        AND rit.Perd_Cal_Yr IN ('2022', '2023', '2024', '2025', '2026')
        and rit.market NOT IN('Estonia','Latvia','Lithuania','Greece_CC')
        AND rit.MAT_Flag IN ('MAT TY','MAT LY','MAT 2LY')
    GROUP BY
         rit.Market,
         rit.Manufacturer
),
Jul26_DatabaseTotals AS ( -- Renamed from Jun26_GrandTotals to reflect per-database totals
    -- This CTE calculates the total value and volume for each Database for the March 2026 period.
    SELECT
        Market, -- Group by Market
        SUM(Value_000_CHF) AS Total_Value_000_CHF,
        SUM(Volume_000) AS Total_Volume_000
    FROM
        Jul26_SegmentAggregates
    GROUP BY
        Market -- Group by Market
),
Jul26_Percentages AS (
    -- This CTE calculates the percentage contribution of each Global OGF Segment *within its Database* for March 2026.
    SELECT
        sa.Market,
        sa.Manufacturer,
        (sa.Value_000_CHF * 100.0 / dt.Total_Value_000_CHF) AS Raw_Value_Percentage_Jul26, -- Uses per-database total
        (sa.Volume_000 * 100.0 / dt.Total_Volume_000) AS Raw_Volume_Percentage_Jul26, -- Uses per-database total
        CONCAT(ROUND((sa.Value_000_CHF * 100.0 / dt.Total_Value_000_CHF), 2), '%') AS `Sum_of_Value_000_CHF_Jul26`,
        CONCAT(ROUND((sa.Volume_000 * 100.0 / dt.Total_Volume_000), 2), '%') AS `Sum_of_Volume_000_Jul26`
    FROM
        Jul26_SegmentAggregates sa
    INNER JOIN -- Changed to INNER JOIN to link segments to their database totals
        Jul26_DatabaseTotals dt ON sa.Market = dt.Market
),
Jun26_SegmentAggregates AS (
    -- This CTE is similar to Jun26_SegmentAggregates but for the February 2026 period, using historical data tables.
    SELECT
        rit.Market as Market,
        rit.Manufacturer as Manufacturer,
        sum(rit.Value_000_CHF) as Value_000_CHF,
        sum(rit.Volume_000) as Volume_000
    FROM
        glbl_cpw_prod.history_conformed.retailindextotal_Jun26 AS rit
     WHERE
        rit.Category IN ('RTE', 'RTE_OTHER')
        AND rit.Perd_Cal_Yr IN ('2022', '2023', '2024', '2025', '2026')
        and rit.market NOT IN('Estonia','Latvia','Lithuania','Greece_CC')
        AND rit.MAT_Flag IN ('MAT TY','MAT LY','MAT 2LY')
    GROUP BY
         rit.Market,
         rit.Manufacturer
),
Jun26_DatabaseTotals AS ( -- Renamed from Jun26_GrandTotals to reflect per-database totals
    -- This CTE calculates the total value and volume for each Database for the February 2026 period.
    SELECT
        Market, -- Group by Market
        SUM(Value_000_CHF) AS Total_Value_000_CHF,
        SUM(Volume_000) AS Total_Volume_000
    FROM
        Jun26_SegmentAggregates
    GROUP BY
        Market -- Group by Database
),
Jun26_Percentages AS (
    -- This CTE calculates the percentage contribution of each Global OGF Segment *within its Database* for February 2026.
    SELECT
        sa.Market,
        sa.Manufacturer,
        (sa.Value_000_CHF * 100.0 / dt.Total_Value_000_CHF) AS Raw_Value_Percentage_Jun26, -- Uses per-database total
        (sa.Volume_000 * 100.0 / dt.Total_Volume_000) AS Raw_Volume_Percentage_Jun26, -- Uses per-database total
        CONCAT(ROUND((sa.Value_000_CHF * 100.0 / dt.Total_Value_000_CHF), 2), '%') AS `Sum_of_Value_000_CHF_Jun26`,
        CONCAT(ROUND((sa.Volume_000 * 100.0 / dt.Total_Volume_000), 2), '%') AS `Sum_of_Volume_000_Jun26`
    FROM
        Jun26_SegmentAggregates sa
    INNER JOIN -- Changed to INNER JOIN to link segments to their database totals
        Jun26_DatabaseTotals dt ON sa.Market = dt.Market
),
Overall_GrandTotals AS ( -- New CTE for overall grand totals across all databases
    SELECT
        SUM(Value_000_CHF) AS Overall_Total_Value_000_CHF_Jul26,
        SUM(Volume_000) AS Overall_Total_Volume_000_Jul26
    FROM
        Jul26_SegmentAggregates
),
Overall_GrandTotals_Feb AS ( -- New CTE for overall grand totals across all databases for Feb
    SELECT
        SUM(Value_000_CHF) AS Overall_Total_Value_000_CHF_Jun26,
        SUM(Volume_000) AS Overall_Total_Volume_000_Jun26
    FROM
        Jun26_SegmentAggregates
)
-- Final SELECT statement to combine the results from March and February 2026 percentages,
-- add a grand total row, and calculate the differences.
SELECT
    COALESCE(mp.Market, fp.Market) AS `Market`,
    COALESCE(mp.Manufacturer, fp.Manufacturer) AS `Global_Manufacturer`,
    mp.`Sum_of_Value_000_CHF_Jul26`,
    mp.`Sum_of_Volume_000_Jul26`,
    fp.`Sum_of_Value_000_CHF_Jun26`,
    fp.`Sum_of_Volume_000_Jun26`,
    CONCAT(ROUND(COALESCE(mp.Raw_Value_Percentage_Jul26, 0) - COALESCE(fp.Raw_Value_Percentage_Jun26, 0), 2), '%') AS `Value difference Jul26 vs Jun26`,
    CONCAT(ROUND(COALESCE(mp.Raw_Volume_Percentage_Jul26, 0) - COALESCE(fp.Raw_Volume_Percentage_Jun26, 0), 2), '%') AS `Volume difference Jul26 vs Jun26`
FROM
    Jul26_Percentages mp
FULL OUTER JOIN
    Jun26_Percentages fp ON mp.Manufacturer = fp.Manufacturer
    AND mp.Market = fp.Market
UNION ALL
-- Grand Total row for overall percentages (across all databases)
SELECT
    'Overall' AS `Market`, -- Label for the overall total
    'Grand Total' AS `Global_Manufacturer`,
    CONCAT(ROUND((SUM(sa.Value_000_CHF) * 100.0 / ogt.Overall_Total_Value_000_CHF_Jul26), 2), '%') AS `Sum_of_Value_000_CHF_Jul26`,
    CONCAT(ROUND((SUM(sa.Volume_000) * 100.0 / ogt.Overall_Total_Volume_000_Jul26), 2), '%') AS `Sum_of_Volume_000_Jun26`,
    CONCAT(ROUND((SUM(sa_mar.Value_000_CHF) * 100.0 / ogt_mar.Overall_Total_Value_000_CHF_Jun26), 2), '%') AS `Sum_of_Value_000_CHF_Jun26`,
    CONCAT(ROUND((SUM(sa_mar.Volume_000) * 100.0 / ogt_mar.Overall_Total_Volume_000_Jun26), 2), '%') AS `Sum_of_Volume_000_Jun26`,
    CONCAT(ROUND(
        (SUM(sa.Value_000_CHF) * 100.0 / ogt.Overall_Total_Value_000_CHF_Jul26) -
        (SUM(sa_mar.Value_000_CHF) * 100.0 / ogt_mar.Overall_Total_Value_000_CHF_Jun26), 2), '%') AS `Value difference Jul26 vs Jun26`,
    CONCAT(ROUND(
        (SUM(sa.Volume_000) * 100.0 / ogt.Overall_Total_Volume_000_Jul26) -
        (SUM(sa_mar.Volume_000) * 100.0 / ogt_mar.Overall_Total_Volume_000_Jun26), 2), '%') AS `Volume difference Jul26 vs Jun26`
FROM
    Jun26_SegmentAggregates sa
CROSS JOIN Overall_GrandTotals ogt
CROSS JOIN Jun26_SegmentAggregates sa_mar
CROSS JOIN Overall_GrandTotals_Feb ogt_mar
GROUP BY
    ogt.Overall_Total_Value_000_CHF_Jul26, ogt.Overall_Total_Volume_000_Jul26,
    ogt_mar.Overall_Total_Value_000_CHF_Jun26, ogt_mar.Overall_Total_Volume_000_Jun26
ORDER BY
    CASE WHEN `Global_Manufacturer` = 'Grand Total' THEN 2 ELSE 1 END,
    `Global_Manufacturer`,
    `Market`;

Market,Global_Manufacturer,Sum_of_Value_000_CHF_Jul26,Sum_of_Volume_000_Jul26,Sum_of_Value_000_CHF_Jun26,Sum_of_Volume_000_Jun26,Value difference Jul26 vs Jun26,Volume difference Jul26 vs Jun26
Portugal,A CENTAZZI,1.40%,0.78%,1.38%,0.77%,0.01%,0.01%
Mexico,AIRES DE CAMPO,0.03%,0.01%,0.03%,0.01%,0.00%,0.00%
Thailand,AKARAWIN INTERFOOD,0.60%,0.68%,0.61%,0.68%,-0.01%,0.00%
Saudi_Arabia,AL MATROOD CO,0.73%,1.02%,0.73%,1.02%,0.00%,-0.01%
Greece,ALARA WHOLEFOODS,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Ireland,ALARA WHOLEFOODS,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Kuwait,ALARA WHOLEFOODS,0.04%,0.05%,0.04%,0.05%,0.00%,0.00%
Malaysia,ALARA WHOLEFOODS,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Portugal,ALARA WHOLEFOODS,0.01%,0.00%,0.01%,0.00%,0.00%,0.00%
Saudi_Arabia,ALARA WHOLEFOODS,0.01%,0.01%,0.01%,0.01%,0.00%,0.00%


In [0]:
%sql
-- Products whose Global_OGF_Segment changed between Jun26 snapshot and current conformed
-- for Spain & Switzerland (RTE / RTE_OTHER)
SELECT
    product_current.Local_Market,
    product_current.product_id,
    product_current.Local_Desc,
    -- product_currrent.Global_SKU,
    -- product_old.Global_SKU,
    product_current.Global_Manufacturer,
    product_old.Global_OGF_Segment  AS OGF_Segment_Jun26,
    product_current.Global_OGF_Segment  AS OGF_Segment_Jul26,
    product_current.Global_Category
FROM glbl_cpw_prod.conformed.dimproduct product_current
JOIN glbl_cpw_prod.history_conformed.dimproduct_Jun26 product_old
  ON product_current.product_id = product_old.product_id
  AND product_current.Local_Market = product_old.Local_Market
WHERE product_current.Global_Category IN ('RTE','RTE_OTHER')
  AND product_current.Local_Market IN ('Spain','Switzerland')
  AND product_current.Global_OGF_Segment <> product_old.Global_OGF_Segment
ORDER BY product_current.Local_Market, product_old.Global_OGF_Segment, product_old.Global_OGF_Segment;

Local_Market,product_id,Local_Desc,Global_Manufacturer,OGF_Segment_Jun26,OGF_Segment_Jul26,Global_Category
Switzerland,25592,M CLASSIC . CEREAL FLAKES CHOCO BTL 01 ER 600 G,PRIVATE LABEL,SIMPLE GOODNESS,EVERYDAY WELLNESS,RTE


# Global_OGF_Segment Product vs Product History

In [0]:
%sql
-- Quantify the sales impact of the reclassified products AND check for products
-- that exist in one snapshot but not the other (new/removed)
WITH reclass AS (
    -- Products whose OGF segment changed
    SELECT p_cur.product_id, p_cur.Local_Market,
           p_old.Global_OGF_Segment AS seg_jun, p_cur.Global_OGF_Segment AS seg_jul
    FROM glbl_cpw_prod.conformed.dimproduct p_cur
    JOIN glbl_cpw_prod.history_conformed.dimproduct_Jun26 p_old
      ON p_cur.product_id = p_old.product_id AND p_cur.Local_Market = p_old.Local_Market
    WHERE p_cur.Global_Category IN ('RTE','RTE_OTHER')
      AND p_cur.Local_Market IN ('Spain','Switzerland')
      AND p_cur.Global_OGF_Segment <> p_old.Global_OGF_Segment
),
jul26_sales AS (
    -- Current conformed sales for the reclassified products (MAT scope)
    SELECT p.Local_Market, p.product_id, p.Global_OGF_Segment,
           SUM(f.Value_000) AS Value_000, SUM(f.Volume_000) AS Volume_000
    FROM glbl_cpw_prod.conformed.factretailsales f
    JOIN glbl_cpw_prod.conformed.dimproduct p ON f.LocalProductKey = p.LocalProductKey
    JOIN glbl_cpw_prod.conformed.dimmarket gm ON f.LocalMarketKey = gm.LocalMarketKey
    JOIN glbl_cpw_prod.adhoc.period_mat_ytd pmt
      ON f.LocalPeriodKey = pmt.LocalPeriodKey AND f.DataProvider = pmt.DataProvider AND f.Database = pmt.Database
    WHERE gm.Global_Total_Mkt_Flag = 'Y'
      AND p.Global_Category IN ('RTE','RTE_OTHER')
      AND p.Local_Market IN ('Spain','Switzerland')
      AND pmt.MAT_Flag IN ('MAT TY','MAT LY','MAT 2LY')
    GROUP BY p.Local_Market, p.product_id, p.Global_OGF_Segment
),
market_totals AS (
    SELECT Local_Market, SUM(Value_000) AS Total_Value, SUM(Volume_000) AS Total_Volume
    FROM jul26_sales GROUP BY Local_Market
)
SELECT r.Local_Market, r.product_id, r.seg_jun, r.seg_jul,
       ROUND(s.Value_000, 2) AS Value_000,
       ROUND(s.Value_000 * 100.0 / mt.Total_Value, 2) AS Pct_of_Market_Value,
       ROUND(s.Volume_000, 2) AS Volume_000,
       ROUND(s.Volume_000 * 100.0 / mt.Total_Volume, 2) AS Pct_of_Market_Volume
FROM reclass r
LEFT JOIN jul26_sales s ON r.product_id = s.product_id AND r.Local_Market = s.Local_Market
LEFT JOIN market_totals mt ON r.Local_Market = mt.Local_Market
ORDER BY r.Local_Market, r.product_id

Local_Market,product_id,seg_jun,seg_jul,Value_000,Pct_of_Market_Value,Volume_000,Pct_of_Market_Volume
Switzerland,25592,SIMPLE GOODNESS,EVERYDAY WELLNESS,0.00,0.00,0.00,0.00


# Switzerland OGF segment Values Jul26 vs Jun26

In [0]:
%sql
-- Compare absolute value/volume by OGF segment for Switzerland: Jul26 vs Jun26
-- to see whether the shift comes from data changes or just the MAT window rolling
WITH jul_26 AS (
    SELECT p.Global_OGF_Segment,
           SUM(f.Value_000) AS Value_000, SUM(f.Volume_000) AS Volume_000
    FROM glbl_cpw_prod.conformed.factretailsales f
    JOIN glbl_cpw_prod.conformed.dimproduct p ON f.LocalProductKey = p.LocalProductKey
    JOIN glbl_cpw_prod.conformed.dimmarket gm ON f.LocalMarketKey = gm.LocalMarketKey
    JOIN glbl_cpw_prod.adhoc.period_mat_ytd pmt
      ON f.LocalPeriodKey = pmt.LocalPeriodKey AND f.DataProvider = pmt.DataProvider AND f.Database = pmt.Database
    WHERE gm.Global_Total_Mkt_Flag = 'Y' AND p.Global_Category IN ('RTE','RTE_OTHER')
      AND gm.Local_Market = 'Switzerland'
      AND pmt.MAT_Flag IN ('MAT TY','MAT LY','MAT 2LY')
    GROUP BY p.Global_OGF_Segment
),
jun_26 AS (
    SELECT p.Global_OGF_Segment,
           SUM(f.Value_000) AS Value_000, SUM(f.Volume_000) AS Volume_000
    FROM glbl_cpw_prod.history_conformed.factretailsales_Jun26 f
    JOIN glbl_cpw_prod.history_conformed.dimproduct_Jun26 p ON f.LocalProductKey = p.LocalProductKey
    JOIN glbl_cpw_prod.history_conformed.dimmarket_Jun26 gm ON f.LocalMarketKey = gm.LocalMarketKey
    JOIN glbl_cpw_prod.history_conformed.period_mat_ytd_Jun26 pmy ON f.LocalPeriodKey = pmy.LocalPeriodKey
    WHERE gm.Global_Total_Mkt_Flag = 'Y' AND p.Global_Category IN ('RTE','RTE_OTHER')
      AND gm.Local_Market = 'Switzerland'
      AND pmy.MAT_Flag IN ('MAT TY','MAT LY','MAT 2LY')
    GROUP BY p.Global_OGF_Segment
)
SELECT COALESCE(j.Global_OGF_Segment, n.Global_OGF_Segment) AS OGF_Segment,
       ROUND(j.Value_000, 2) AS Jul26_Value, ROUND(n.Value_000, 2) AS Jun26_Value,
       ROUND(j.Value_000 - n.Value_000, 2) AS Value_Diff,
       ROUND(j.Volume_000, 2) AS Jul26_Volume, ROUND(n.Volume_000, 2) AS Jun26_Volume,
       ROUND(j.Volume_000 - n.Volume_000, 2) AS Volume_Diff
FROM jul_26 j FULL OUTER JOIN jun_26 n ON j.Global_OGF_Segment = n.Global_OGF_Segment
ORDER BY OGF_Segment

OGF_Segment,Jul26_Value,Jun26_Value,Value_Diff,Jul26_Volume,Jun26_Volume,Volume_Diff
CHILDHOOD FUN,43488.44,43605.17,-116.73,4765.18,4790.89,-25.72
EVERYDAY WELLNESS,22120.71,22186.09,-65.38,1879.52,1885.84,-6.32
NATURALLY DELICIOUS,151927.50,130267.99,21659.51,16647.48,13647.08,3000.40
SIMPLE GOODNESS,146814.06,168233.16,-21419.10,16508.71,19507.97,-2999.26
TASTY FAVOURITES,51082.07,51045.79,36.28,4809.07,4791.80,17.26
UNCLASSIFIED,8763.20,8742.87,20.33,1002.45,1008.18,-5.74


In [0]:
%sql
select p.product_id,f.value_000,f.volume_000 from glbl_cpw_prod.history_conformed.factretailsales_jun26 f
inner join glbl_cpw_prod.conformed.dimproduct p on p.LocalProductKey=f.LocalProductKey
where p.product_id='25592';

product_id,value_000,volume_000
25592,0.0000,0.0000
25592,0.0000,0.0000
25592,0.0000,0.0000
25592,0.0000,0.0000
25592,0.0000,0.0000
25592,0.0000,0.0000
25592,0.0000,0.0000
25592,0.0000,0.0000
25592,0.0000,0.0000
25592,0.0000,0.0000


In [0]:
%sql
-- Compare absolute value/volume by OGF segment for Switzerland: Jul26 vs Jun26
-- to see whether the shift comes from data changes or just the MAT window rolling
WITH jul_26 AS (
    SELECT p.Global_OGF_Segment,
           SUM(f.Value_000) AS Value_000, SUM(f.Volume_000) AS Volume_000
    FROM glbl_cpw_prod.conformed.factretailsales f
    JOIN glbl_cpw_prod.conformed.dimproduct p ON f.LocalProductKey = p.LocalProductKey
    JOIN glbl_cpw_prod.conformed.dimmarket gm ON f.LocalMarketKey = gm.LocalMarketKey
    JOIN glbl_cpw_prod.adhoc.period_mat_ytd pmt
      ON f.LocalPeriodKey = pmt.LocalPeriodKey AND f.DataProvider = pmt.DataProvider AND f.Database = pmt.Database
    WHERE gm.Global_Total_Mkt_Flag = 'Y' AND p.Global_Category IN ('RTE','RTE_OTHER')
      AND gm.Local_Market = 'Switzerland'
      AND pmt.MAT_Flag IN ('MAT TY','MAT LY','MAT 2LY')
    GROUP BY p.Global_OGF_Segment
),
jun_26 AS (
    SELECT p.Global_OGF_Segment,
           SUM(f.Value_000) AS Value_000, SUM(f.Volume_000) AS Volume_000
    FROM glbl_cpw_prod.history_conformed.factretailsales_Jun26 f
    JOIN glbl_cpw_prod.history_conformed.dimproduct_Jun26 p ON f.LocalProductKey = p.LocalProductKey
    JOIN glbl_cpw_prod.history_conformed.dimmarket_Jun26 gm ON f.LocalMarketKey = gm.LocalMarketKey
    JOIN glbl_cpw_prod.history_conformed.period_mat_ytd_Jun26 pmy ON f.LocalPeriodKey = pmy.LocalPeriodKey
    WHERE gm.Global_Total_Mkt_Flag = 'Y' AND p.Global_Category IN ('RTE','RTE_OTHER')
      AND gm.Local_Market = 'Switzerland'
      AND pmy.MAT_Flag IN ('MAT TY','MAT LY','MAT 2LY')
    GROUP BY p.Global_OGF_Segment
)
SELECT COALESCE(j.Global_OGF_Segment, n.Global_OGF_Segment) AS OGF_Segment,
       ROUND(j.Value_000, 2) AS Jul26_Value, ROUND(n.Value_000, 2) AS Jun26_Value,
       ROUND(j.Value_000 - n.Value_000, 2) AS Value_Diff,
       ROUND(j.Volume_000, 2) AS Jul26_Volume, ROUND(n.Volume_000, 2) AS Jun26_Volume,
       ROUND(j.Volume_000 - n.Volume_000, 2) AS Volume_Diff
FROM jul_26 j FULL OUTER JOIN jun_26 n ON j.Global_OGF_Segment = n.Global_OGF_Segment
ORDER BY OGF_Segment

OGF_Segment,Jul26_Value,Jun26_Value,Value_Diff,Jul26_Volume,Jun26_Volume,Volume_Diff
CHILDHOOD FUN,43488.44,43605.17,-116.73,4765.18,4790.89,-25.72
EVERYDAY WELLNESS,22120.71,22186.09,-65.38,1879.52,1885.84,-6.32
NATURALLY DELICIOUS,151927.50,130267.99,21659.51,16647.48,13647.08,3000.40
SIMPLE GOODNESS,146814.06,168233.16,-21419.10,16508.71,19507.97,-2999.26
TASTY FAVOURITES,51082.07,51045.79,36.28,4809.07,4791.80,17.26
UNCLASSIFIED,8763.20,8742.87,20.33,1002.45,1008.18,-5.74
